In [ ]:
import fsspec
import geopandas as gpd


fs = fsspec.filesystem('s3', anon=True)
with fs.open('s3://pism-cloud-data/terra/rgi.gpkg') as rgi_file:
    gdf = gpd.read_file(rgi_file)

gdf = gdf.sort_values("area_km2", ascending=False)

# Alaska glaciers larger than 500 km^2
# https://glims-rgi.github.io/rgi_user_guide/02_regions_definition.html
rgis = gdf[(gdf["o1region"] == "01") & (gdf["area_km2"] >= 500)]

In [ ]:
# Commented out lines will be set when we prepare the pism-cloud AWS job
STAGE_TEMPLATE =     {
    # "name": "RGI2000-v7.0-C-01-09429_era5_agu_1year",
    "job_type": "PISM_TERRA_PREP_ENSEMBLE",
    "job_parameters": {
        # "rgi_id": "RGI2000-v7.0-C-01-09429",
        "rgi_gpkg": "s3://pism-cloud-data/terra/rgi.gpkg",
        "pism_config": "s3://pism-cloud-data/terra/era5_ec2.toml",
        "run_template": "s3://pism-cloud-data/terra/ec2.j2",
        "uq_config": "s3://pism-cloud-data/terra/era5_agu.toml",
        # "ntasks": 32,
    }
}

In [ ]:
from copy import deepcopy
from pathlib import Path


prepared_jobs = []
for rgi_id in rgis.rgi_id:
    job_dict = deepcopy(STAGE_TEMPLATE)
    job_dict['name'] = f'{rgi_id}_{Path(job_dict["job_parameters"]["pism_config"]).stem}'
    job_dict['job_parameters']['rgi_id'] = rgi_id
    job_dict['job_parameters']['ntasks'] = 32
    prepared_jobs.append(job_dict)


In [ ]:
import hyp3_sdk as sdk

hyp3 = sdk.HyP3('https://pism-cloud-test.asf.alaska.edu', prompt=True)

jobs = hyp3.submit_prepared_jobs(prepared_jobs)
print(jobs)

job_names = {job.name for job in jobs}
print(job_names)

In [ ]:
jobs = hyp3.watch(jobs)

In [ ]:
import s3fs

PISM_CLOUD_BUCKET = 'hyp3-pism-cloud-test-contentbucket-zs9dctrqrlvx'

def get_run_scripts(job: sdk.Job) ->  list[str]:
    fs = s3fs.S3FileSystem(anon=True)
    files = fs.ls(f'{PISM_CLOUD_BUCKET}/{job.job_id}/{job.job_parameters["rgi_id"]}/run_scripts')
    return [str(Path(file).relative_to(f'{PISM_CLOUD_BUCKET}/{job.job_id}/')) for file in files]


In [ ]:
EXECUTE_TEMPLATE = {
    # "name": "RGI2000-v7.0-C-01-09429_era5_agu_1year",
    "job_type": "PISM_TERRA_EXECUTE",
    "job_parameters": {
        # "ensemble_job_id": "042ffcdc-2134-4b18-b1af-b22fdf7cbb52",
        # "run_script": "RGI2000-v7.0-C-01-09429/run_scripts/submit_g400m_RGI2000-v7.0-C-01-09429_id_0_1978-01-01_1979-01-01.sh",
    }
}

prepared_jobs = []
for job in jobs:
    run_scripts = get_run_scripts(job)
    for script in run_scripts:
        job_dict = deepcopy(EXECUTE_TEMPLATE)
        job_dict['name'] = job.name
        job_dict['job_parameters']['ensemble_job_id'] = job.job_id
        job_dict['job_parameters']['run_script'] = script
        prepared_jobs.append(job_dict)


In [ ]:
hyp3 = sdk.HyP3('https://pism-cloud-test.asf.alaska.edu', prompt=True)

jobs = hyp3.submit_prepared_jobs(prepared_jobs)
print(jobs)

job_names = {job.name for job in jobs}
print(job_names)